# Task #34 (Story #6): Tính tỉ lệ mất cân bằng tổng thể (2 phương án xử lý NA)

Đọc `data/processed/orders_labeled.csv`, ép kiểu lại các cột, rồi tính tỉ lệ mất cân bằng của `is_delayed` theo 2 phương án đã chốt cùng User:
- **Phương án A (chính)**: loại nhóm NA, mẫu số = số đơn xác định được (đúng hạn/trễ) — dùng cho mục tiêu mô hình hóa.
- **Phương án B (phụ)**: gộp NA thành nhóm rủi ro thứ 3, mẫu số = toàn bộ dataset — dùng cho giám sát vận hành.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/orders_labeled.csv", low_memory=False)

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

bool_cols = [
    "payment_has_boleto",
    "payment_has_credit_card",
    "payment_has_debit_card",
    "payment_has_not_defined",
    "payment_has_voucher",
    "items_multi_seller",
]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

df["is_delayed"] = df["is_delayed"].astype("boolean")

df.shape

(99441, 42)

## Phương án A (chính): loại nhóm NA, mẫu số = số đơn xác định được

In [2]:
n_delayed = (df["is_delayed"] == True).sum()
n_on_time = (df["is_delayed"] == False).sum()
n_determined = n_delayed + n_on_time

ratio_a = pd.DataFrame({
    "n_orders": [n_on_time, n_delayed],
    "pct": [n_on_time / n_determined, n_delayed / n_determined],
}, index=["Đúng hạn", "Trễ"])

print(f"Mẫu số (đơn xác định được): {n_determined}")
ratio_a["pct"] = (ratio_a["pct"] * 100).round(2)
ratio_a

Mẫu số (đơn xác định được): 96476


,n_orders,pct
Đúng hạn,88649,91.89
Trễ,7827,8.11


## Phương án B (phụ): gộp NA thành nhóm rủi ro thứ 3, mẫu số = toàn bộ dataset

In [3]:
n_total = len(df)
n_na = df["is_delayed"].isna().sum()

ratio_b = pd.DataFrame({
    "n_orders": [n_on_time, n_delayed, n_na],
    "pct": [n_on_time / n_total, n_delayed / n_total, n_na / n_total],
}, index=["Đúng hạn", "Trễ", "Chưa xác định (NA)"])

print(f"Mẫu số (toàn bộ dataset): {n_total}")
ratio_b["pct"] = (ratio_b["pct"] * 100).round(2)
ratio_b

Mẫu số (toàn bộ dataset): 99441


,n_orders,pct
Đúng hạn,88649,89.15
Trễ,7827,7.87
Chưa xác định (NA),2965,2.98


## Kết luận

- **Phương án A** (mẫu số 96.476 đơn xác định được): lớp thiểu số "Trễ" chiếm ~8,11% — đây là tỉ lệ mất cân bằng dùng làm căn cứ chọn kỹ thuật xử lý mất cân bằng khi huấn luyện mô hình (Story #8).
- **Phương án B** (mẫu số toàn bộ 99.441 đơn): nhóm "Chưa xác định" chiếm ~2,98%, phần lớn là đơn `shipped`/`canceled`/`unavailable` — giữ để giám sát vận hành, không dùng để tính tỉ lệ mất cân bằng cho mô hình.